# Results tables for the paper

This notebook reads saved experiment results and displays the main performance table, performance tables by training size, and separate calibration and multiclass tables. It does not train models, recalculate metrics, or modify the input pickle files. Only load pickle files from a trusted source.

The reference setting uses 1,000 training observations for Heart, Diabetes-H, Diabetes-130, Obesity, and Obesity-Bin, and 455 for Breast Cancer. The training size includes any internal validation data and excludes the held-out test set.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Edit these selections as needed.
DATASETS = ['heart', 'diabetes_h', 'diabetes_130', 'obesity', 'obesity_bin', 'breast_cancer']
MODELS = ['mlp', 'lr', 'rf', 'nam', 'kan', 'kan_gam', 'ebm', 'xgboost', 'lightgbm']
DECIMALS = 3
HIGHLIGHT_BEST = True
SHOW_COVERAGE = True

DATASET_LABELS = {
    'heart': 'Heart', 'diabetes_h': 'Diabetes-H', 'diabetes_130': 'Diabetes-130',
    'obesity': 'Obesity', 'obesity_bin': 'Obesity-Bin', 'breast_cancer': 'Breast Cancer',
}
MODEL_LABELS = {
    'mlp': 'MLP', 'lr': 'Logistic regression', 'rf': 'Random forest', 'nam': 'NAM',
    'kan': 'Logistic-KAN', 'kan_gam': 'KAAM', 'ebm': 'EBM',
    'xgboost': 'XGBoost', 'lightgbm': 'LightGBM',
}
DATASET_SIZES = {
    'heart': [100, 500, 1000, 5000, 10000],
    'diabetes_h': [100, 500, 1000, 5000, 10000],
    'diabetes_130': [100, 500, 1000, 5000, 10000],
    'obesity': [100, 500, 1000, 1688],
    'obesity_bin': [100, 500, 1000, 1688],
    'breast_cancer': [100, 455],
}
REFERENCE_SIZES = {dataset: (455 if dataset == 'breast_cancer' else 1000) for dataset in DATASETS}
MULTICLASS_DATASETS = {'diabetes_h', 'diabetes_130', 'obesity'}

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'main_performance.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the notebook from the project or one of its subdirectories.')
RESULTS_DIR = PROJECT_ROOT / 'results_metrics' / 'current'
print(f'Reading results from: {RESULTS_DIR}')

Reading results from: C:\Users\Alex\CODE\classification_with_kans\results_metrics\current


## Reading and interpreting the tables

Values are the saved **mean across valid bootstrap replicates**, followed by the saved 95% percentile interval, `mean (lower–upper)`. These means need not equal the metric calculated once on the complete test set. The bootstrap resamples test observations with replacement while keeping predictions and the trained model fixed; it does not measure variability from retraining or changing the train/test split. Replicates lacking a class are discarded. Coverage below flags results with fewer valid replicates than requested.

**Bold** marks the best saved, unrounded mean for the same dataset and metric, including exact ties. Higher is better for classification metrics and lower is better for ECE, Brier score and log loss. Bold is descriptive: it does not indicate statistical significance or use the confidence intervals to test differences. Classwise precision and recall lists are not highlighted.

In [2]:
MAIN_METRICS = {'Accuracy': 'accuracy', 'ROC-AUC': 'roc_auc', 'F1': 'f1', 'Precision': 'precision', 'Recall': 'recall'}
CALIBRATION_METRICS = {'ECE': None, 'Brier score': 'brier', 'Log loss': 'log_loss'}
MULTICLASS_METRICS = {'Macro-F1': 'f1_macro', 'Balanced accuracy': 'balanced_accuracy'}
TABLES = {}


def metric_pair(result, key):
    try:
        mean = float(result['metric_mean'][key])
        lower, upper = (float(v) for v in result['metric_ci'][key])
        if not np.isfinite([mean, lower, upper]).all() or lower > upper:
            return None
        return mean, lower, upper
    except (KeyError, TypeError, ValueError):
        return None


def format_metric(result, key):
    values = metric_pair(result, key)
    if values is None:
        return 'N/A'
    mean, lower, upper = values
    return f'{mean:.{DECIMALS}f} ({lower:.{DECIMALS}f}–{upper:.{DECIMALS}f})'


def load_results():
    results, issues = {}, []
    for dataset in DATASETS:
        for size in DATASET_SIZES[dataset]:
            for model in MODELS:
                path = RESULTS_DIR / str(size) / dataset / f'{model}.pkl'
                identifier = (dataset, size, model)
                if not path.is_file():
                    issues.append((*identifier, 'Missing file', str(path)))
                    continue
                try:
                    with path.open('rb') as file:
                        result = pickle.load(file)
                    if not isinstance(result, dict):
                        raise ValueError('Expected a result dictionary')
                    if any(result.get(k) != v for k, v in {'dataset': dataset, 'train_size': size, 'model': model}.items()):
                        raise ValueError('Dataset, size or model metadata does not match the path')
                    if result.get('bootstrap', {}).get('ci') != 95:
                        raise ValueError('A saved 95% bootstrap interval is not available')
                    if result.get('bootstrap', {}).get('method') != 'percentile':
                        raise ValueError('The saved bootstrap method is not marked percentile')
                    classes = np.asarray(result['classes']).reshape(-1)
                    if len(classes) < 2:
                        raise ValueError('Fewer than two class labels')
                except Exception as exc:
                    issues.append((*identifier, f'Unreadable or incompatible: {exc}', str(path)))
                    continue
                results[identifier] = result
                required = list(MAIN_METRICS.values()) + ['brier', 'log_loss', 'ece' if len(classes) == 2 else 'ece_macro']
                if len(classes) > 2:
                    required += list(MULTICLASS_METRICS.values())
                    required += [f'{metric}_class_{i}' for metric in ('precision', 'recall', 'ece') for i in range(len(classes))]
                missing = [key for key in required if metric_pair(result, key) is None]
                if missing:
                    issues.append((*identifier, 'Missing or invalid metrics: ' + ', '.join(missing), str(path)))
                bootstrap = result['bootstrap']
                if bootstrap.get('n_valid') is not None and bootstrap.get('n_bootstrap') is not None and bootstrap['n_valid'] < bootstrap['n_bootstrap']:
                    issues.append((*identifier, f"Valid bootstrap replicates: {bootstrap['n_valid']}/{bootstrap['n_bootstrap']}", str(path)))
    return results, pd.DataFrame(issues, columns=['Dataset', 'Training size', 'Model', 'Issue', 'File'])


results, issues = load_results()


def selected_results(setting):
    return [(dataset, size, model, results[(dataset, size, model)])
            for dataset, size in setting.items() for model in MODELS
            if (dataset, size, model) in results]


def size_setting(size):
    return {dataset: size for dataset in DATASETS if size in DATASET_SIZES[dataset]}


def class_orders(setting):
    orders = {}
    for dataset, _, _, result in selected_results(setting):
        classes = tuple(np.asarray(result['classes']).reshape(-1).tolist())
        if dataset in orders and orders[dataset] != classes:
            raise ValueError(f'Inconsistent class order for {dataset}: {orders[dataset]} vs {classes}')
        orders[dataset] = classes
    return orders


def make_table(kind, setting):
    rows, raw = [], []
    orders = class_orders(setting)
    for dataset, size, model, result in selected_results(setting):
        classes = orders[dataset]
        base = {'Dataset': DATASET_LABELS.get(dataset, dataset), 'Training size': size,
                'Model': MODEL_LABELS.get(model, model)}
        if kind == 'main':
            columns = MAIN_METRICS
        elif kind == 'calibration':
            base['ECE definition'] = f'Positive class: {classes[1]}' if len(classes) == 2 else 'Classwise macro'
            columns = {**CALIBRATION_METRICS, 'ECE': 'ece' if len(classes) == 2 else 'ece_macro'}
        elif kind == 'multiclass':
            if len(classes) <= 2:
                continue
            columns = MULTICLASS_METRICS
        elif kind == 'classwise_ece':
            if len(classes) <= 2:
                continue
            for i, label in enumerate(classes):
                key = f'ece_class_{i}'
                pair = metric_pair(result, key)
                rows.append({**base, 'Class': str(label), 'ECE': format_metric(result, key)})
                raw.append({'Dataset': base['Dataset'], 'Class': str(label), 'ECE': pair[0] if pair else np.nan})
            continue
        else:
            raise ValueError(f'Unknown table type: {kind}')
        numeric = {'Dataset': base['Dataset']}
        for label, key in columns.items():
            pair = metric_pair(result, key)
            base[label] = format_metric(result, key)
            numeric[label] = pair[0] if pair else np.nan
        if kind == 'multiclass':
            for name in ('precision', 'recall'):
                base[name.capitalize() + ' by class'] = ' / '.join(
                    format_metric(result, f'{name}_class_{i}') for i in range(len(classes))
                )
        rows.append(base)
        raw.append(numeric)
    return pd.DataFrame(rows), pd.DataFrame(raw), orders


def show_table(name, kind, setting):
    frame, numeric, orders = make_table(kind, setting)
    if frame.empty:
        print('No valid saved results are available for this setting.')
        return
    styles = pd.DataFrame('', index=frame.index, columns=frame.columns)
    if HIGHLIGHT_BEST:
        metric_columns = list(MAIN_METRICS) if kind == 'main' else (
            list(CALIBRATION_METRICS) if kind == 'calibration' else (
                list(MULTICLASS_METRICS) if kind == 'multiclass' else ['ECE']
            )
        )
        grouping = ['Dataset', 'Class'] if kind == 'classwise_ece' else ['Dataset']
        for column in metric_columns:
            groups = numeric.groupby(grouping, sort=False)[column]
            best = groups.transform('min' if kind in ('calibration', 'classwise_ece') else 'max')
            styles.loc[numeric[column].notna() & numeric[column].eq(best), column] = 'font-weight: bold'
    styled = frame.style.apply(lambda _: styles, axis=None).hide(axis='index')
    TABLES[name] = styled
    display(styled)
    if kind == 'multiclass':
        for dataset, classes in orders.items():
            if len(classes) > 2:
                print(f"{DATASET_LABELS.get(dataset, dataset)} class order: {' / '.join(map(str, classes))}")


def latex_table(name):
    """Return LaTeX for a table that has already been displayed."""
    return TABLES[name].to_latex(hrules=True, convert_css=True)

##### Result coverage and input diagnostics

Coverage counts files with compatible metadata and a saved 95% percentile interval. A compatible file can still have missing individual metrics, which are listed in the diagnostics and shown as `N/A`. Fewer valid than requested bootstrap replicates are also reported.

In [3]:
if SHOW_COVERAGE:
    coverage = []
    for dataset in DATASETS:
        for size in DATASET_SIZES[dataset]:
            available = [model for model in MODELS if (dataset, size, model) in results]
            coverage.append({'Dataset': DATASET_LABELS.get(dataset, dataset), 'Training size': size,
                             'Available': f'{len(available)}/{len(MODELS)}',
                             'Missing or incompatible': ', '.join(m for m in MODELS if m not in available) or '—'})
    display(pd.DataFrame(coverage))
    if issues.empty:
        print('No input issues or reduced valid bootstrap counts detected.')
    else:
        display(issues)

,Dataset,Training size,Available,Missing or incompatible
0,Heart,100,9/9,—
1,Heart,500,9/9,—
2,Heart,1000,9/9,—
3,Heart,5000,9/9,—
4,Heart,10000,9/9,—
5,Diabetes-H,100,9/9,—
6,Diabetes-H,500,9/9,—
7,Diabetes-H,1000,9/9,—
8,Diabetes-H,5000,9/9,—
9,Diabetes-H,10000,9/9,—


No input issues or reduced valid bootstrap counts detected.


## Main classification performance

These tables contain accuracy, ROC-AUC, F1, precision and recall. Binary precision, recall and F1 use the positive class; multiclass precision, recall and F1 are support-weighted. Multiclass ROC-AUC is support-weighted one-vs-one. Bold indicates the best unrounded mean within each dataset and metric; it is not a significance test.

##### Main performance — N = 1,000; Breast Cancer N = 455

The paper's reference comparison includes all selected datasets and models. Training sizes refer to train plus any internal validation data.

In [4]:
show_table('main_reference', 'main', REFERENCE_SIZES)

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,1000,MLP,0.900 (0.889–0.912),0.718 (0.687–0.750),0.023 (0.000–0.051),0.149 (0.000–0.318),0.013 (0.000–0.028)
Heart,1000,Logistic regression,0.749 (0.731–0.765),0.820 (0.796–0.845),0.354 (0.314–0.390),0.233 (0.201–0.261),0.731 (0.673–0.785)
Heart,1000,Random forest,0.865 (0.852–0.877),0.803 (0.778–0.827),0.327 (0.275–0.379),0.307 (0.254–0.364),0.350 (0.290–0.411)
Heart,1000,NAM,0.906 (0.895–0.917),0.370 (0.334–0.406),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,1000,Logistic-KAN,0.758 (0.741–0.775),0.833 (0.810–0.856),0.370 (0.330–0.406),0.245 (0.214–0.275),0.757 (0.702–0.810)
Heart,1000,KAAM,0.759 (0.744–0.776),0.835 (0.811–0.856),0.365 (0.323–0.401),0.243 (0.211–0.272),0.736 (0.682–0.789)
Heart,1000,EBM,0.908 (0.896–0.918),0.831 (0.809–0.854),0.094 (0.047–0.146),0.603 (0.385–0.833),0.051 (0.025–0.082)
Heart,1000,XGBoost,0.892 (0.880–0.904),0.793 (0.765–0.819),0.182 (0.125–0.241),0.320 (0.218–0.417),0.127 (0.086–0.172)
Heart,1000,LightGBM,0.900 (0.889–0.911),0.798 (0.772–0.824),0.198 (0.142–0.260),0.409 (0.296–0.524),0.131 (0.091–0.178)
Diabetes-H,1000,MLP,0.828 (0.812–0.842),0.666 (0.628–0.704),0.791 (0.771–0.809),0.773 (0.751–0.794),0.828 (0.812–0.842)


##### Main performance — training size N = 100

This table includes only datasets configured for N = 100. Bold indicates the highest saved mean for each dataset and metric.

In [5]:
show_table('main_100', 'main', size_setting(100))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,100,MLP,0.828 (0.813–0.842),0.574 (0.533–0.612),0.166 (0.127–0.210),0.153 (0.115–0.196),0.183 (0.138–0.235)
Heart,100,Logistic regression,0.731 (0.714–0.749),0.801 (0.775–0.826),0.338 (0.298–0.372),0.220 (0.190–0.248),0.731 (0.673–0.787)
Heart,100,Random forest,0.873 (0.860–0.886),0.787 (0.758–0.814),0.227 (0.176–0.283),0.265 (0.201–0.331),0.200 (0.150–0.253)
Heart,100,NAM,0.906 (0.895–0.917),0.422 (0.383–0.461),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,100,Logistic-KAN,0.816 (0.801–0.830),0.755 (0.726–0.783),0.286 (0.243–0.331),0.225 (0.188–0.266),0.392 (0.333–0.453)
Heart,100,KAAM,0.786 (0.769–0.803),0.779 (0.750–0.807),0.320 (0.278–0.363),0.228 (0.194–0.265),0.536 (0.472–0.606)
Heart,100,EBM,0.892 (0.880–0.903),0.793 (0.769–0.816),0.075 (0.037–0.119),0.192 (0.100–0.297),0.047 (0.023–0.076)
Heart,100,XGBoost,0.880 (0.867–0.892),0.766 (0.737–0.794),0.181 (0.130–0.238),0.253 (0.179–0.338),0.141 (0.098–0.189)
Heart,100,LightGBM,0.873 (0.861–0.887),0.760 (0.731–0.790),0.222 (0.169–0.276),0.263 (0.199–0.333),0.192 (0.145–0.244)
Diabetes-H,100,MLP,0.076 (0.065–0.086),0.568 (0.534–0.605),0.092 (0.077–0.107),0.743 (0.675–0.802),0.076 (0.065–0.086)


##### Main performance — training size N = 455

This table includes only datasets configured for N = 455. Bold indicates the highest saved mean for each dataset and metric.

In [6]:
show_table('main_455', 'main', size_setting(455))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Breast Cancer,455,MLP,0.957 (0.921–0.991),0.990 (0.973–1.000),0.966 (0.935–0.993),0.959 (0.908–1.000),0.973 (0.931–1.000)
Breast Cancer,455,Logistic regression,0.957 (0.912–0.991),0.994 (0.982–1.000),0.965 (0.931–0.993),0.972 (0.929–1.000),0.959 (0.910–1.000)
Breast Cancer,455,Random forest,0.930 (0.886–0.974),0.963 (0.914–0.998),0.944 (0.902–0.978),0.958 (0.903–1.000),0.931 (0.871–0.986)
Breast Cancer,455,NAM,0.903 (0.851–0.956),0.957 (0.916–0.987),0.921 (0.871–0.966),0.942 (0.883–0.986),0.903 (0.829–0.971)
Breast Cancer,455,Logistic-KAN,0.957 (0.912–0.991),0.987 (0.964–1.000),0.965 (0.929–0.993),0.972 (0.928–1.000),0.959 (0.907–1.000)
Breast Cancer,455,KAAM,0.956 (0.912–0.991),0.995 (0.985–1.000),0.965 (0.929–0.993),0.958 (0.905–1.000),0.973 (0.929–1.000)
Breast Cancer,455,EBM,0.974 (0.939–1.000),0.992 (0.978–1.000),0.980 (0.953–1.000),0.973 (0.930–1.000),0.987 (0.953–1.000)
Breast Cancer,455,XGBoost,0.957 (0.912–0.991),0.984 (0.956–1.000),0.965 (0.929–0.993),0.972 (0.928–1.000),0.959 (0.907–1.000)
Breast Cancer,455,LightGBM,0.939 (0.886–0.982),0.983 (0.956–0.999),0.950 (0.904–0.984),0.971 (0.925–1.000),0.931 (0.868–0.986)


##### Main performance — training size N = 500

This table includes only datasets configured for N = 500. Bold indicates the highest saved mean for each dataset and metric.

In [7]:
show_table('main_500', 'main', size_setting(500))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,500,MLP,0.896 (0.884–0.908),0.694 (0.662–0.727),0.132 (0.081–0.183),0.304 (0.194–0.422),0.085 (0.050–0.120)
Heart,500,Logistic regression,0.752 (0.735–0.770),0.794 (0.768–0.819),0.346 (0.310–0.383),0.230 (0.201–0.261),0.699 (0.638–0.757)
Heart,500,Random forest,0.875 (0.863–0.887),0.790 (0.763–0.817),0.315 (0.261–0.370),0.325 (0.263–0.387),0.306 (0.252–0.364)
Heart,500,NAM,0.906 (0.895–0.917),0.417 (0.378–0.454),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,500,Logistic-KAN,0.780 (0.765–0.796),0.805 (0.779–0.829),0.349 (0.306–0.387),0.242 (0.207–0.273),0.627 (0.566–0.688)
Heart,500,KAAM,0.774 (0.758–0.790),0.804 (0.779–0.827),0.352 (0.311–0.390),0.241 (0.208–0.273),0.656 (0.596–0.715)
Heart,500,EBM,0.908 (0.897–0.919),0.813 (0.789–0.838),0.154 (0.098–0.216),0.585 (0.412–0.750),0.089 (0.054–0.129)
Heart,500,XGBoost,0.890 (0.878–0.902),0.770 (0.744–0.798),0.221 (0.161–0.281),0.332 (0.245–0.417),0.167 (0.119–0.215)
Heart,500,LightGBM,0.892 (0.880–0.903),0.771 (0.745–0.798),0.205 (0.150–0.263),0.331 (0.240–0.417),0.149 (0.107–0.197)
Diabetes-H,500,MLP,0.809 (0.794–0.825),0.586 (0.545–0.623),0.773 (0.753–0.793),0.768 (0.742–0.793),0.809 (0.794–0.825)


##### Main performance — training size N = 1,000

This table includes only datasets configured for N = 1,000. Bold indicates the highest saved mean for each dataset and metric.

In [8]:
show_table('main_1000', 'main', size_setting(1000))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,1000,MLP,0.900 (0.889–0.912),0.718 (0.687–0.750),0.023 (0.000–0.051),0.149 (0.000–0.318),0.013 (0.000–0.028)
Heart,1000,Logistic regression,0.749 (0.731–0.765),0.820 (0.796–0.845),0.354 (0.314–0.390),0.233 (0.201–0.261),0.731 (0.673–0.785)
Heart,1000,Random forest,0.865 (0.852–0.877),0.803 (0.778–0.827),0.327 (0.275–0.379),0.307 (0.254–0.364),0.350 (0.290–0.411)
Heart,1000,NAM,0.906 (0.895–0.917),0.370 (0.334–0.406),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,1000,Logistic-KAN,0.758 (0.741–0.775),0.833 (0.810–0.856),0.370 (0.330–0.406),0.245 (0.214–0.275),0.757 (0.702–0.810)
Heart,1000,KAAM,0.759 (0.744–0.776),0.835 (0.811–0.856),0.365 (0.323–0.401),0.243 (0.211–0.272),0.736 (0.682–0.789)
Heart,1000,EBM,0.908 (0.896–0.918),0.831 (0.809–0.854),0.094 (0.047–0.146),0.603 (0.385–0.833),0.051 (0.025–0.082)
Heart,1000,XGBoost,0.892 (0.880–0.904),0.793 (0.765–0.819),0.182 (0.125–0.241),0.320 (0.218–0.417),0.127 (0.086–0.172)
Heart,1000,LightGBM,0.900 (0.889–0.911),0.798 (0.772–0.824),0.198 (0.142–0.260),0.409 (0.296–0.524),0.131 (0.091–0.178)
Diabetes-H,1000,MLP,0.828 (0.812–0.842),0.666 (0.628–0.704),0.791 (0.771–0.809),0.773 (0.751–0.794),0.828 (0.812–0.842)


##### Main performance — training size N = 1,688

This table includes only datasets configured for N = 1,688. Bold indicates the highest saved mean for each dataset and metric.

In [9]:
show_table('main_1688', 'main', size_setting(1688))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Obesity,1688,MLP,0.860 (0.825–0.894),0.981 (0.973–0.988),0.859 (0.824–0.892),0.863 (0.829–0.895),0.860 (0.825–0.894)
Obesity,1688,Logistic regression,0.728 (0.686–0.771),0.935 (0.921–0.947),0.714 (0.666–0.759),0.730 (0.683–0.775),0.728 (0.686–0.771)
Obesity,1688,Random forest,0.977 (0.960–0.991),0.999 (0.997–1.000),0.977 (0.960–0.991),0.978 (0.962–0.991),0.977 (0.960–0.991)
Obesity,1688,NAM,0.853 (0.818–0.887),0.981 (0.974–0.987),0.851 (0.816–0.885),0.854 (0.818–0.887),0.853 (0.818–0.887)
Obesity,1688,Logistic-KAN,0.964 (0.946–0.981),0.998 (0.997–1.000),0.965 (0.946–0.981),0.966 (0.949–0.981),0.964 (0.946–0.981)
Obesity,1688,KAAM,0.955 (0.936–0.974),0.996 (0.993–0.999),0.955 (0.936–0.974),0.958 (0.941–0.974),0.955 (0.936–0.974)
Obesity,1688,EBM,0.969 (0.953–0.983),0.999 (0.997–1.000),0.969 (0.953–0.983),0.970 (0.954–0.984),0.969 (0.953–0.983)
Obesity,1688,XGBoost,0.991 (0.981–0.998),1.000 (1.000–1.000),0.991 (0.981–0.998),0.991 (0.982–0.998),0.991 (0.981–0.998)
Obesity,1688,LightGBM,0.991 (0.981–0.998),1.000 (0.999–1.000),0.991 (0.981–0.998),0.991 (0.983–0.998),0.991 (0.981–0.998)
Obesity-Bin,1688,MLP,0.974 (0.957–0.988),0.995 (0.989–0.999),0.972 (0.953–0.987),0.984 (0.964–1.000),0.960 (0.930–0.984)


##### Main performance — training size N = 5,000

This table includes only datasets configured for N = 5,000. Bold indicates the highest saved mean for each dataset and metric.

In [10]:
show_table('main_5000', 'main', size_setting(5000))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,5000,MLP,0.904 (0.892–0.915),0.798 (0.772–0.822),0.070 (0.026–0.115),0.392 (0.176–0.600),0.038 (0.014–0.064)
Heart,5000,Logistic regression,0.756 (0.738–0.772),0.850 (0.829–0.871),0.379 (0.339–0.417),0.249 (0.218–0.279),0.791 (0.737–0.844)
Heart,5000,Random forest,0.853 (0.841–0.867),0.829 (0.804–0.853),0.395 (0.344–0.443),0.323 (0.275–0.369),0.511 (0.448–0.574)
Heart,5000,NAM,0.906 (0.895–0.917),0.788 (0.759–0.816),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,5000,Logistic-KAN,0.742 (0.724–0.759),0.850 (0.829–0.871),0.374 (0.336–0.409),0.242 (0.214–0.271),0.821 (0.769–0.870)
Heart,5000,KAAM,0.752 (0.734–0.768),0.849 (0.827–0.870),0.378 (0.339–0.415),0.248 (0.217–0.277),0.804 (0.753–0.852)
Heart,5000,EBM,0.907 (0.896–0.918),0.845 (0.823–0.866),0.176 (0.121–0.239),0.532 (0.394–0.674),0.106 (0.070–0.149)
Heart,5000,XGBoost,0.902 (0.891–0.913),0.811 (0.783–0.836),0.255 (0.192–0.320),0.452 (0.344–0.557),0.178 (0.132–0.228)
Heart,5000,LightGBM,0.907 (0.896–0.918),0.835 (0.811–0.859),0.204 (0.143–0.268),0.518 (0.383–0.648),0.128 (0.087–0.172)
Diabetes-H,5000,MLP,0.835 (0.821–0.850),0.747 (0.720–0.774),0.794 (0.774–0.814),0.779 (0.757–0.801),0.835 (0.821–0.850)


##### Main performance — training size N = 10,000

This table includes only datasets configured for N = 10,000. Bold indicates the highest saved mean for each dataset and metric.

In [11]:
show_table('main_10000', 'main', size_setting(10000))

Dataset,Training size,Model,Accuracy,ROC-AUC,F1,Precision,Recall
Heart,10000,MLP,0.909 (0.898–0.919),0.846 (0.825–0.868),0.129 (0.078–0.187),0.629 (0.450–0.812),0.072 (0.042–0.107)
Heart,10000,Logistic regression,0.756 (0.738–0.772),0.852 (0.831–0.873),0.383 (0.343–0.422),0.251 (0.220–0.283),0.804 (0.752–0.850)
Heart,10000,Random forest,0.855 (0.841–0.869),0.839 (0.815–0.860),0.416 (0.366–0.461),0.335 (0.287–0.381),0.550 (0.486–0.613)
Heart,10000,NAM,0.906 (0.895–0.917),0.818 (0.795–0.843),0.000 (0.000–0.000),0.000 (0.000–0.000),0.000 (0.000–0.000)
Heart,10000,Logistic-KAN,0.743 (0.725–0.760),0.850 (0.829–0.871),0.376 (0.338–0.412),0.244 (0.215–0.272),0.825 (0.777–0.871)
Heart,10000,KAAM,0.749 (0.731–0.766),0.851 (0.829–0.871),0.380 (0.341–0.415),0.248 (0.218–0.276),0.816 (0.764–0.865)
Heart,10000,EBM,0.909 (0.898–0.920),0.852 (0.831–0.872),0.166 (0.110–0.225),0.604 (0.442–0.756),0.097 (0.062–0.134)
Heart,10000,XGBoost,0.906 (0.895–0.918),0.825 (0.799–0.849),0.271 (0.206–0.337),0.503 (0.400–0.607),0.186 (0.135–0.239)
Heart,10000,LightGBM,0.906 (0.896–0.917),0.845 (0.822–0.867),0.162 (0.107–0.220),0.508 (0.364–0.658),0.097 (0.063–0.134)
Diabetes-H,10000,MLP,0.843 (0.828–0.858),0.734 (0.705–0.765),0.803 (0.784–0.822),0.794 (0.772–0.815),0.843 (0.828–0.858)


## Mathematical definitions

For class $k$, let $TP_k$, $FP_k$, $FN_k$ and $n_k$ denote true positives, false positives, false negatives and class support. For $n$ observations and $K$ classes:

$$\mathrm{Accuracy}=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(\hat y_i=y_i),\quad
P_k=\frac{TP_k}{TP_k+FP_k},\quad R_k=\frac{TP_k}{TP_k+FN_k},\quad
F1_k=\frac{2P_kR_k}{P_k+R_k}.$$

Zero-denominator precision, recall and F1 are set to zero. For multiclass tasks, the main table reports $M_{\mathrm{weighted}}=\sum_k(n_k/n)M_k$ for precision, recall and F1. For single-label multiclass classification weighted recall equals accuracy. The complementary table reports:

$$F1_{\mathrm{macro}}=\frac{1}{K}\sum_k F1_k,\qquad
\mathrm{Balanced\ accuracy}=\frac{1}{K}\sum_k R_k.$$

Macro-F1 weights classes equally; weighted F1 gives greater weight to frequent classes. Per-class precision and recall retain the class-specific information. Binary precision, recall and F1 use the second saved class as the positive class.

ROC-AUC measures the area under the receiver operating characteristic curve. In multiclass tasks the implementation uses scikit-learn's support-weighted one-vs-one ROC-AUC.

For class $k$, let $p_{ik}$ be its predicted probability and $y_{ik}=\mathbf{1}(y_i=k)$. With equally spaced probability bins $B_b$, classwise ECE is:

$$\mathrm{ECE}_k=\sum_b\frac{|B_b|}{n}\left|\frac{1}{|B_b|}\sum_{i\in B_b} y_{ik}-\frac{1}{|B_b|}\sum_{i\in B_b}p_{ik}\right|.$$

Empty bins contribute zero. Binary ECE uses the positive class; multiclass ECE is $K^{-1}\sum_k\mathrm{ECE}_k$. ECE depends on the binning scheme.

The stored binary Brier score is $n^{-1}\sum_i(p_{i+}-y_{i+})^2$. The multiclass implementation uses $n^{-1}\sum_i\sum_k(p_{ik}-y_{ik})^2$; these conventions have different scales. Log loss is:

$$\mathrm{Log\ loss}=-\frac{1}{n}\sum_i\sum_k y_{ik}\log(p_{ik}).$$

Probabilities are clipped to avoid $\log(0)$. Lower ECE, Brier score and log loss are better; Brier and log loss also reflect discrimination, so they are not pure measures of calibration.

## Calibration tables

ECE uses the positive class for binary tasks and the macro-average of classwise ECE for multiclass tasks. Classwise ECE tables show each multiclass label separately. Bold marks the lowest unrounded mean within a dataset (and class for classwise ECE). Results are shown as saved bootstrap means and 95% intervals.

##### Calibration summary — N = 1,000; Breast Cancer N = 455

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [12]:
show_table('calibration_reference', 'calibration', REFERENCE_SIZES)

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,1000,MLP,Positive class: 1,0.144 (0.133–0.154),0.102 (0.097–0.108),0.362 (0.349–0.374)
Heart,1000,Logistic regression,Positive class: 1,0.246 (0.233–0.258),0.167 (0.158–0.177),0.500 (0.477–0.524)
Heart,1000,Random forest,Positive class: 1,0.120 (0.109–0.131),0.100 (0.094–0.105),0.348 (0.314–0.394)
Heart,1000,NAM,Positive class: 1,0.094 (0.082–0.105),0.094 (0.083–0.105),0.783 (0.691–0.878)
Heart,1000,Logistic-KAN,Positive class: 1,0.254 (0.242–0.266),0.160 (0.151–0.167),0.482 (0.462–0.502)
Heart,1000,KAAM,Positive class: 1,0.257 (0.245–0.268),0.160 (0.152–0.168),0.483 (0.464–0.503)
Heart,1000,EBM,Positive class: 1,0.012 (0.005–0.021),0.073 (0.066–0.080),0.247 (0.226–0.268)
Heart,1000,XGBoost,Positive class: 1,0.073 (0.063–0.084),0.088 (0.079–0.098),0.366 (0.323–0.411)
Heart,1000,LightGBM,Positive class: 1,0.062 (0.051–0.073),0.084 (0.075–0.092),0.329 (0.290–0.368)
Diabetes-H,1000,MLP,Classwise macro,0.069 (0.062–0.077),0.276 (0.260–0.293),0.517 (0.492–0.543)


##### Classwise ECE — N = 1,000; Breast Cancer N = 455

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [13]:
show_table('classwise_ece_reference', 'classwise_ece', REFERENCE_SIZES)

Dataset,Training size,Model,Class,ECE
Diabetes-H,1000,MLP,0,0.099 (0.085–0.113)
Diabetes-H,1000,MLP,1,0.087 (0.081–0.093)
Diabetes-H,1000,MLP,2,0.021 (0.012–0.030)
Diabetes-H,1000,Logistic regression,0,0.220 (0.207–0.234)
Diabetes-H,1000,Logistic regression,1,0.150 (0.143–0.156)
Diabetes-H,1000,Logistic regression,2,0.070 (0.057–0.082)
Diabetes-H,1000,Random forest,0,0.066 (0.052–0.079)
Diabetes-H,1000,Random forest,1,0.012 (0.008–0.016)
Diabetes-H,1000,Random forest,2,0.061 (0.048–0.073)
Diabetes-H,1000,NAM,0,0.021 (0.012–0.030)


##### Calibration summary — training size N = 100

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [14]:
show_table('calibration_100', 'calibration', size_setting(100))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,100,MLP,Positive class: 1,0.285 (0.273–0.296),0.173 (0.169–0.177),0.531 (0.522–0.540)
Heart,100,Logistic regression,Positive class: 1,0.278 (0.265–0.290),0.176 (0.168–0.184),0.524 (0.506–0.544)
Heart,100,Random forest,Positive class: 1,0.115 (0.104–0.125),0.097 (0.091–0.102),0.336 (0.311–0.372)
Heart,100,NAM,Positive class: 1,0.094 (0.082–0.105),0.094 (0.083–0.105),0.782 (0.690–0.875)
Heart,100,Logistic-KAN,Positive class: 1,0.194 (0.182–0.206),0.132 (0.126–0.139),0.420 (0.405–0.436)
Heart,100,KAAM,Positive class: 1,0.223 (0.211–0.234),0.145 (0.138–0.152),0.450 (0.434–0.467)
Heart,100,EBM,Positive class: 1,0.061 (0.050–0.071),0.086 (0.077–0.095),0.321 (0.284–0.359)
Heart,100,XGBoost,Positive class: 1,0.076 (0.065–0.088),0.095 (0.086–0.104),0.365 (0.327–0.405)
Heart,100,LightGBM,Positive class: 1,0.092 (0.080–0.104),0.099 (0.090–0.109),0.451 (0.398–0.510)
Diabetes-H,100,MLP,Classwise macro,0.445 (0.436–0.454),1.072 (1.058–1.086),1.824 (1.795–1.857)


##### Classwise ECE — training size N = 100

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [15]:
show_table('classwise_ece_100', 'classwise_ece', size_setting(100))

Dataset,Training size,Model,Class,ECE
Diabetes-H,100,MLP,0,0.662 (0.647–0.676)
Diabetes-H,100,MLP,1,0.547 (0.539–0.556)
Diabetes-H,100,MLP,2,0.127 (0.114–0.140)
Diabetes-H,100,Logistic regression,0,0.298 (0.284–0.313)
Diabetes-H,100,Logistic regression,1,0.186 (0.180–0.192)
Diabetes-H,100,Logistic regression,2,0.112 (0.098–0.125)
Diabetes-H,100,Random forest,0,0.037 (0.025–0.049)
Diabetes-H,100,Random forest,1,0.017 (0.012–0.022)
Diabetes-H,100,Random forest,2,0.045 (0.033–0.057)
Diabetes-H,100,NAM,0,0.041 (0.029–0.053)


##### Calibration summary — training size N = 455

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [16]:
show_table('calibration_455', 'calibration', size_setting(455))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Breast Cancer,455,MLP,Positive class: 1,0.051 (0.029–0.080),0.029 (0.010–0.055),0.107 (0.053–0.182)
Breast Cancer,455,Logistic regression,Positive class: 1,0.052 (0.032–0.076),0.028 (0.013–0.049),0.107 (0.065–0.159)
Breast Cancer,455,Random forest,Positive class: 1,0.073 (0.043–0.110),0.047 (0.023–0.081),0.732 (0.092–1.701)
Breast Cancer,455,NAM,Positive class: 1,0.093 (0.051–0.137),0.083 (0.044–0.127),0.348 (0.167–0.567)
Breast Cancer,455,Logistic-KAN,Positive class: 1,0.055 (0.033–0.083),0.036 (0.015–0.064),0.137 (0.071–0.222)
Breast Cancer,455,KAAM,Positive class: 1,0.037 (0.016–0.066),0.026 (0.007–0.051),0.083 (0.029–0.152)
Breast Cancer,455,EBM,Positive class: 1,0.043 (0.019–0.075),0.028 (0.009–0.056),0.097 (0.034–0.188)
Breast Cancer,455,XGBoost,Positive class: 1,0.049 (0.022–0.085),0.038 (0.012–0.073),0.166 (0.047–0.328)
Breast Cancer,455,LightGBM,Positive class: 1,0.062 (0.025–0.112),0.056 (0.019–0.102),0.324 (0.097–0.649)


##### Classwise ECE — training size N = 455

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [17]:
show_table('classwise_ece_455', 'classwise_ece', size_setting(455))

No valid saved results are available for this setting.


##### Calibration summary — training size N = 500

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [18]:
show_table('calibration_500', 'calibration', size_setting(500))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,500,MLP,Positive class: 1,0.239 (0.227–0.249),0.139 (0.135–0.143),0.457 (0.449–0.465)
Heart,500,Logistic regression,Positive class: 1,0.244 (0.231–0.257),0.169 (0.161–0.178),0.507 (0.484–0.531)
Heart,500,Random forest,Positive class: 1,0.118 (0.106–0.129),0.097 (0.092–0.103),0.322 (0.307–0.335)
Heart,500,NAM,Positive class: 1,0.094 (0.082–0.105),0.094 (0.083–0.105),0.778 (0.685–0.869)
Heart,500,Logistic-KAN,Positive class: 1,0.247 (0.235–0.259),0.157 (0.149–0.164),0.482 (0.463–0.501)
Heart,500,KAAM,Positive class: 1,0.251 (0.239–0.263),0.160 (0.152–0.168),0.489 (0.470–0.509)
Heart,500,EBM,Positive class: 1,0.021 (0.012–0.031),0.075 (0.067–0.082),0.254 (0.232–0.276)
Heart,500,XGBoost,Positive class: 1,0.079 (0.068–0.090),0.091 (0.082–0.101),0.394 (0.348–0.440)
Heart,500,LightGBM,Positive class: 1,0.083 (0.072–0.094),0.093 (0.084–0.103),0.467 (0.408–0.524)
Diabetes-H,500,MLP,Classwise macro,0.142 (0.133–0.151),0.356 (0.341–0.370),0.651 (0.629–0.672)


##### Classwise ECE — training size N = 500

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [19]:
show_table('classwise_ece_500', 'classwise_ece', size_setting(500))

Dataset,Training size,Model,Class,ECE
Diabetes-H,500,MLP,0,0.207 (0.192–0.221)
Diabetes-H,500,MLP,1,0.148 (0.141–0.154)
Diabetes-H,500,MLP,2,0.070 (0.056–0.083)
Diabetes-H,500,Logistic regression,0,0.221 (0.208–0.235)
Diabetes-H,500,Logistic regression,1,0.144 (0.137–0.151)
Diabetes-H,500,Logistic regression,2,0.078 (0.065–0.090)
Diabetes-H,500,Random forest,0,0.040 (0.028–0.052)
Diabetes-H,500,Random forest,1,0.010 (0.005–0.015)
Diabetes-H,500,Random forest,2,0.039 (0.028–0.052)
Diabetes-H,500,NAM,0,0.016 (0.008–0.025)


##### Calibration summary — training size N = 1,000

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [20]:
show_table('calibration_1000', 'calibration', size_setting(1000))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,1000,MLP,Positive class: 1,0.144 (0.133–0.154),0.102 (0.097–0.108),0.362 (0.349–0.374)
Heart,1000,Logistic regression,Positive class: 1,0.246 (0.233–0.258),0.167 (0.158–0.177),0.500 (0.477–0.524)
Heart,1000,Random forest,Positive class: 1,0.120 (0.109–0.131),0.100 (0.094–0.105),0.348 (0.314–0.394)
Heart,1000,NAM,Positive class: 1,0.094 (0.082–0.105),0.094 (0.083–0.105),0.783 (0.691–0.878)
Heart,1000,Logistic-KAN,Positive class: 1,0.254 (0.242–0.266),0.160 (0.151–0.167),0.482 (0.462–0.502)
Heart,1000,KAAM,Positive class: 1,0.257 (0.245–0.268),0.160 (0.152–0.168),0.483 (0.464–0.503)
Heart,1000,EBM,Positive class: 1,0.012 (0.005–0.021),0.073 (0.066–0.080),0.247 (0.226–0.268)
Heart,1000,XGBoost,Positive class: 1,0.073 (0.063–0.084),0.088 (0.079–0.098),0.366 (0.323–0.411)
Heart,1000,LightGBM,Positive class: 1,0.062 (0.051–0.073),0.084 (0.075–0.092),0.329 (0.290–0.368)
Diabetes-H,1000,MLP,Classwise macro,0.069 (0.062–0.077),0.276 (0.260–0.293),0.517 (0.492–0.543)


##### Classwise ECE — training size N = 1,000

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [21]:
show_table('classwise_ece_1000', 'classwise_ece', size_setting(1000))

Dataset,Training size,Model,Class,ECE
Diabetes-H,1000,MLP,0,0.099 (0.085–0.113)
Diabetes-H,1000,MLP,1,0.087 (0.081–0.093)
Diabetes-H,1000,MLP,2,0.021 (0.012–0.030)
Diabetes-H,1000,Logistic regression,0,0.220 (0.207–0.234)
Diabetes-H,1000,Logistic regression,1,0.150 (0.143–0.156)
Diabetes-H,1000,Logistic regression,2,0.070 (0.057–0.082)
Diabetes-H,1000,Random forest,0,0.066 (0.052–0.079)
Diabetes-H,1000,Random forest,1,0.012 (0.008–0.016)
Diabetes-H,1000,Random forest,2,0.061 (0.048–0.073)
Diabetes-H,1000,NAM,0,0.021 (0.012–0.030)


##### Calibration summary — training size N = 1,688

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [22]:
show_table('calibration_1688', 'calibration', size_setting(1688))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Obesity,1688,MLP,Classwise macro,0.041 (0.036–0.047),0.215 (0.183–0.250),0.439 (0.379–0.504)
Obesity,1688,Logistic regression,Classwise macro,0.079 (0.072–0.087),0.469 (0.442–0.497),0.949 (0.900–1.000)
Obesity,1688,Random forest,Classwise macro,0.041 (0.037–0.045),0.091 (0.076–0.108),0.224 (0.195–0.254)
Obesity,1688,NAM,Classwise macro,0.059 (0.053–0.065),0.259 (0.231–0.286),0.544 (0.497–0.588)
Obesity,1688,Logistic-KAN,Classwise macro,0.019 (0.016–0.023),0.059 (0.041–0.078),0.133 (0.100–0.169)
Obesity,1688,KAAM,Classwise macro,0.029 (0.025–0.033),0.090 (0.071–0.113),0.196 (0.161–0.240)
Obesity,1688,EBM,Classwise macro,0.025 (0.022–0.029),0.064 (0.050–0.079),0.142 (0.120–0.165)
Obesity,1688,XGBoost,Classwise macro,0.007 (0.005–0.009),0.018 (0.008–0.031),0.044 (0.022–0.074)
Obesity,1688,LightGBM,Classwise macro,0.004 (0.001–0.006),0.018 (0.005–0.034),0.062 (0.008–0.137)
Obesity-Bin,1688,MLP,Positive class: 1,0.040 (0.029–0.051),0.024 (0.015–0.034),0.096 (0.070–0.126)


##### Classwise ECE — training size N = 1,688

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [23]:
show_table('classwise_ece_1688', 'classwise_ece', size_setting(1688))

Dataset,Training size,Model,Class,ECE
Obesity,1688,MLP,0,0.037 (0.027–0.050)
Obesity,1688,MLP,1,0.054 (0.038–0.072)
Obesity,1688,MLP,2,0.051 (0.035–0.068)
Obesity,1688,MLP,3,0.051 (0.036–0.068)
Obesity,1688,MLP,4,0.061 (0.048–0.074)
Obesity,1688,MLP,5,0.028 (0.020–0.036)
Obesity,1688,MLP,6,0.006 (0.004–0.010)
Obesity,1688,Logistic regression,0,0.087 (0.067–0.107)
Obesity,1688,Logistic regression,1,0.056 (0.038–0.077)
Obesity,1688,Logistic regression,2,0.077 (0.056–0.102)


##### Calibration summary — training size N = 5,000

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [24]:
show_table('calibration_5000', 'calibration', size_setting(5000))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,5000,MLP,Positive class: 1,0.077 (0.067–0.087),0.083 (0.077–0.089),0.294 (0.279–0.309)
Heart,5000,Logistic regression,Positive class: 1,0.256 (0.244–0.268),0.165 (0.156–0.174),0.495 (0.474–0.518)
Heart,5000,Random forest,Positive class: 1,0.138 (0.127–0.149),0.103 (0.097–0.109),0.330 (0.314–0.346)
Heart,5000,NAM,Positive class: 1,0.050 (0.039–0.061),0.086 (0.076–0.095),0.313 (0.282–0.345)
Heart,5000,Logistic-KAN,Positive class: 1,0.263 (0.251–0.275),0.168 (0.159–0.177),0.494 (0.473–0.516)
Heart,5000,KAAM,Positive class: 1,0.267 (0.255–0.279),0.166 (0.158–0.175),0.499 (0.479–0.520)
Heart,5000,EBM,Positive class: 1,0.013 (0.007–0.021),0.071 (0.064–0.078),0.239 (0.217–0.261)
Heart,5000,XGBoost,Positive class: 1,0.044 (0.034–0.055),0.078 (0.070–0.086),0.283 (0.250–0.315)
Heart,5000,LightGBM,Positive class: 1,0.023 (0.014–0.032),0.073 (0.066–0.081),0.247 (0.225–0.271)
Diabetes-H,5000,MLP,Classwise macro,0.017 (0.012–0.023),0.234 (0.217–0.251),0.417 (0.389–0.449)


##### Classwise ECE — training size N = 5,000

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [25]:
show_table('classwise_ece_5000', 'classwise_ece', size_setting(5000))

Dataset,Training size,Model,Class,ECE
Diabetes-H,5000,MLP,0,0.024 (0.015–0.035)
Diabetes-H,5000,MLP,1,0.004 (0.001–0.009)
Diabetes-H,5000,MLP,2,0.023 (0.014–0.033)
Diabetes-H,5000,Logistic regression,0,0.218 (0.205–0.231)
Diabetes-H,5000,Logistic regression,1,0.161 (0.155–0.167)
Diabetes-H,5000,Logistic regression,2,0.058 (0.045–0.070)
Diabetes-H,5000,Random forest,0,0.187 (0.174–0.200)
Diabetes-H,5000,Random forest,1,0.073 (0.066–0.079)
Diabetes-H,5000,Random forest,2,0.114 (0.101–0.126)
Diabetes-H,5000,NAM,0,0.021 (0.012–0.033)


##### Calibration summary — training size N = 10,000

ECE, Brier score and log loss are reported for all selected tasks available in this setting. Lower values are better.

In [26]:
show_table('calibration_10000', 'calibration', size_setting(10000))

Dataset,Training size,Model,ECE definition,ECE,Brier score,Log loss
Heart,10000,MLP,Positive class: 1,0.011 (0.005–0.019),0.071 (0.064–0.078),0.238 (0.219–0.259)
Heart,10000,Logistic regression,Positive class: 1,0.254 (0.242–0.266),0.164 (0.155–0.172),0.490 (0.469–0.513)
Heart,10000,Random forest,Positive class: 1,0.138 (0.127–0.149),0.104 (0.098–0.109),0.327 (0.313–0.342)
Heart,10000,NAM,Positive class: 1,0.126 (0.116–0.136),0.092 (0.087–0.098),0.339 (0.327–0.352)
Heart,10000,Logistic-KAN,Positive class: 1,0.261 (0.249–0.272),0.166 (0.158–0.175),0.492 (0.472–0.514)
Heart,10000,KAAM,Positive class: 1,0.264 (0.252–0.275),0.165 (0.157–0.174),0.493 (0.473–0.514)
Heart,10000,EBM,Positive class: 1,0.014 (0.007–0.023),0.070 (0.063–0.077),0.236 (0.216–0.257)
Heart,10000,XGBoost,Positive class: 1,0.033 (0.023–0.042),0.074 (0.066–0.082),0.259 (0.232–0.287)
Heart,10000,LightGBM,Positive class: 1,0.015 (0.008–0.023),0.071 (0.064–0.078),0.240 (0.219–0.263)
Diabetes-H,10000,MLP,Classwise macro,0.021 (0.015–0.027),0.236 (0.219–0.254),0.425 (0.397–0.456)


##### Classwise ECE — training size N = 10,000

Each row refers to one class of a multiclass task. Lower values are better. Binary tasks are excluded.

In [27]:
show_table('classwise_ece_10000', 'classwise_ece', size_setting(10000))

Dataset,Training size,Model,Class,ECE
Diabetes-H,10000,MLP,0,0.023 (0.014–0.033)
Diabetes-H,10000,MLP,1,0.012 (0.007–0.017)
Diabetes-H,10000,MLP,2,0.028 (0.017–0.039)
Diabetes-H,10000,Logistic regression,0,0.220 (0.207–0.233)
Diabetes-H,10000,Logistic regression,1,0.168 (0.162–0.173)
Diabetes-H,10000,Logistic regression,2,0.054 (0.042–0.065)
Diabetes-H,10000,Random forest,0,0.246 (0.233–0.259)
Diabetes-H,10000,Random forest,1,0.113 (0.107–0.119)
Diabetes-H,10000,Random forest,2,0.134 (0.121–0.146)
Diabetes-H,10000,NAM,0,0.023 (0.012–0.034)


## Complementary multiclass classification tables

These tables contain macro-F1, balanced accuracy, and classwise precision and recall. They are separate from the main performance and calibration tables. Higher is better. Bold identifies the highest unrounded mean for macro-F1 and balanced accuracy; classwise lists are left unhighlighted. The label order is printed beneath each table and must match across models for a dataset.

##### Multiclass metrics — N = 1,000; Breast Cancer N = 455

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [28]:
show_table('multiclass_reference', 'multiclass', REFERENCE_SIZES)

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,1000,MLP,0.372 (0.354–0.389),0.369 (0.356–0.381),0.856 (0.841–0.869) / 0.000 (0.000–0.000) / 0.377 (0.301–0.467),0.959 (0.950–0.968) / 0.000 (0.000–0.000) / 0.147 (0.111–0.185)
Diabetes-H,1000,Logistic regression,0.447 (0.424–0.472),0.457 (0.429–0.489),0.908 (0.896–0.921) / 0.034 (0.000–0.079) / 0.384 (0.338–0.431),0.869 (0.855–0.882) / 0.066 (0.000–0.145) / 0.437 (0.385–0.489)
Diabetes-H,1000,Random forest,0.370 (0.352–0.388),0.368 (0.355–0.382),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.329 (0.250–0.402),0.953 (0.944–0.962) / 0.000 (0.000–0.000) / 0.152 (0.113–0.190)
Diabetes-H,1000,NAM,0.371 (0.353–0.388),0.368 (0.357–0.380),0.855 (0.842–0.870) / 0.000 (0.000–0.000) / 0.540 (0.429–0.659),0.984 (0.979–0.990) / 0.000 (0.000–0.000) / 0.121 (0.087–0.155)
Diabetes-H,1000,Logistic-KAN,0.384 (0.366–0.402),0.377 (0.365–0.390),0.860 (0.846–0.874) / 0.000 (0.000–0.000) / 0.497 (0.404–0.591),0.977 (0.970–0.983) / 0.000 (0.000–0.000) / 0.155 (0.118–0.195)
Diabetes-H,1000,KAAM,0.384 (0.366–0.402),0.377 (0.365–0.390),0.860 (0.846–0.874) / 0.000 (0.000–0.000) / 0.497 (0.404–0.591),0.977 (0.970–0.983) / 0.000 (0.000–0.000) / 0.155 (0.118–0.195)
Diabetes-H,1000,EBM,0.380 (0.362–0.399),0.375 (0.362–0.388),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.553 (0.442–0.660),0.983 (0.977–0.989) / 0.000 (0.000–0.000) / 0.141 (0.106–0.180)
Diabetes-H,1000,XGBoost,0.401 (0.383–0.418),0.395 (0.379–0.410),0.870 (0.856–0.884) / 0.000 (0.000–0.000) / 0.380 (0.318–0.441),0.937 (0.926–0.948) / 0.000 (0.000–0.000) / 0.247 (0.201–0.293)
Diabetes-H,1000,LightGBM,0.402 (0.385–0.420),0.395 (0.381–0.410),0.869 (0.854–0.883) / 0.000 (0.000–0.000) / 0.412 (0.348–0.482),0.949 (0.939–0.958) / 0.000 (0.000–0.000) / 0.236 (0.192–0.281)
Diabetes-130,1000,MLP,0.328 (0.313–0.344),0.347 (0.334–0.361),0.559 (0.537–0.584) / 0.383 (0.344–0.420) / 0.051 (0.000–0.117),0.738 (0.714–0.761) / 0.292 (0.262–0.321) / 0.011 (0.000–0.024)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2
Obesity class order: 0 / 1 / 2 / 3 / 4 / 5 / 6


##### Multiclass metrics — training size N = 100

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [29]:
show_table('multiclass_100', 'multiclass', size_setting(100))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,100,MLP,0.102 (0.088–0.115),0.327 (0.286–0.369),0.850 (0.773–0.918) / 0.017 (0.012–0.022) / 0.190 (0.151–0.231),0.040 (0.031–0.048) / 0.738 (0.619–0.860) / 0.204 (0.166–0.245)
Diabetes-H,100,Logistic regression,0.421 (0.397–0.446),0.451 (0.413–0.491),0.890 (0.876–0.905) / 0.050 (0.020–0.087) / 0.310 (0.265–0.354),0.819 (0.802–0.835) / 0.174 (0.077–0.286) / 0.359 (0.310–0.415)
Diabetes-H,100,Random forest,0.312 (0.304–0.321),0.334 (0.330–0.339),0.842 (0.828–0.856) / 0.000 (0.000–0.000) / 0.156 (0.036–0.290),0.987 (0.982–0.992) / 0.000 (0.000–0.000) / 0.015 (0.003–0.029)
Diabetes-H,100,NAM,0.324 (0.312–0.337),0.342 (0.336–0.348),0.845 (0.831–0.859) / 0.000 (0.000–0.000) / 0.426 (0.222–0.619),0.993 (0.990–0.997) / 0.000 (0.000–0.000) / 0.032 (0.014–0.052)
Diabetes-H,100,Logistic-KAN,0.324 (0.313–0.336),0.342 (0.336–0.348),0.845 (0.831–0.859) / 0.000 (0.000–0.000) / 0.425 (0.240–0.640),0.993 (0.990–0.997) / 0.000 (0.000–0.000) / 0.032 (0.014–0.049)
Diabetes-H,100,KAAM,0.324 (0.313–0.336),0.342 (0.336–0.348),0.845 (0.831–0.859) / 0.000 (0.000–0.000) / 0.425 (0.240–0.640),0.993 (0.990–0.997) / 0.000 (0.000–0.000) / 0.032 (0.014–0.049)
Diabetes-H,100,EBM,0.305 (0.302–0.307),0.333 (0.333–0.333),0.842 (0.828–0.856) / 0.000 (0.000–0.000) / 0.000 (0.000–0.000),1.000 (1.000–1.000) / 0.000 (0.000–0.000) / 0.000 (0.000–0.000)
Diabetes-H,100,XGBoost,0.401 (0.365–0.440),0.387 (0.362–0.416),0.859 (0.845–0.873) / 0.203 (0.000–0.429) / 0.309 (0.238–0.377),0.944 (0.933–0.953) / 0.066 (0.000–0.143) / 0.153 (0.116–0.193)
Diabetes-H,100,LightGBM,0.378 (0.360–0.395),0.373 (0.359–0.388),0.861 (0.846–0.875) / 0.000 (0.000–0.000) / 0.335 (0.266–0.404),0.937 (0.926–0.947) / 0.000 (0.000–0.000) / 0.183 (0.142–0.226)
Diabetes-130,100,MLP,0.301 (0.283–0.318),0.324 (0.302–0.347),0.516 (0.484–0.545) / 0.341 (0.305–0.380) / 0.111 (0.090–0.131),0.421 (0.395–0.448) / 0.229 (0.202–0.256) / 0.322 (0.269–0.378)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2
Obesity class order: 0 / 1 / 2 / 3 / 4 / 5 / 6


##### Multiclass metrics — training size N = 455

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [30]:
show_table('multiclass_455', 'multiclass', size_setting(455))

No valid saved results are available for this setting.


##### Multiclass metrics — training size N = 500

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [31]:
show_table('multiclass_500', 'multiclass', size_setting(500))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,500,MLP,0.349 (0.331–0.371),0.355 (0.335–0.381),0.851 (0.836–0.865) / 0.025 (0.000–0.063) / 0.364 (0.250–0.479),0.948 (0.940–0.958) / 0.045 (0.000–0.116) / 0.072 (0.047–0.101)
Diabetes-H,500,Logistic regression,0.439 (0.417–0.463),0.459 (0.429–0.492),0.910 (0.898–0.922) / 0.040 (0.009–0.082) / 0.344 (0.299–0.389),0.840 (0.825–0.856) / 0.088 (0.020–0.178) / 0.448 (0.393–0.500)
Diabetes-H,500,Random forest,0.376 (0.358–0.394),0.372 (0.359–0.385),0.857 (0.843–0.871) / 0.000 (0.000–0.000) / 0.413 (0.329–0.509),0.967 (0.958–0.975) / 0.000 (0.000–0.000) / 0.149 (0.110–0.189)
Diabetes-H,500,NAM,0.372 (0.353–0.391),0.369 (0.357–0.382),0.855 (0.841–0.870) / 0.000 (0.000–0.000) / 0.546 (0.432–0.661),0.984 (0.978–0.989) / 0.000 (0.000–0.000) / 0.123 (0.089–0.161)
Diabetes-H,500,Logistic-KAN,0.384 (0.364–0.402),0.377 (0.364–0.390),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.531 (0.425–0.636),0.979 (0.972–0.985) / 0.000 (0.000–0.000) / 0.152 (0.111–0.191)
Diabetes-H,500,KAAM,0.384 (0.364–0.402),0.377 (0.364–0.390),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.531 (0.425–0.636),0.979 (0.972–0.985) / 0.000 (0.000–0.000) / 0.152 (0.111–0.191)
Diabetes-H,500,EBM,0.396 (0.377–0.416),0.386 (0.373–0.401),0.862 (0.847–0.876) / 0.000 (0.000–0.000) / 0.540 (0.448–0.639),0.975 (0.968–0.982) / 0.000 (0.000–0.000) / 0.184 (0.144–0.226)
Diabetes-H,500,XGBoost,0.410 (0.385–0.441),0.402 (0.382–0.424),0.871 (0.857–0.885) / 0.098 (0.000–0.333) / 0.352 (0.292–0.411),0.923 (0.911–0.935) / 0.022 (0.000–0.075) / 0.261 (0.214–0.304)
Diabetes-H,500,LightGBM,0.403 (0.386–0.422),0.400 (0.384–0.416),0.871 (0.857–0.885) / 0.000 (0.000–0.000) / 0.369 (0.308–0.427),0.929 (0.917–0.939) / 0.000 (0.000–0.000) / 0.270 (0.223–0.317)
Diabetes-130,500,MLP,0.361 (0.345–0.378),0.370 (0.356–0.385),0.594 (0.572–0.618) / 0.408 (0.374–0.443) / 0.091 (0.039–0.148),0.711 (0.687–0.735) / 0.364 (0.334–0.396) / 0.036 (0.015–0.060)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2
Obesity class order: 0 / 1 / 2 / 3 / 4 / 5 / 6


##### Multiclass metrics — training size N = 1,000

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [32]:
show_table('multiclass_1000', 'multiclass', size_setting(1000))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,1000,MLP,0.372 (0.354–0.389),0.369 (0.356–0.381),0.856 (0.841–0.869) / 0.000 (0.000–0.000) / 0.377 (0.301–0.467),0.959 (0.950–0.968) / 0.000 (0.000–0.000) / 0.147 (0.111–0.185)
Diabetes-H,1000,Logistic regression,0.447 (0.424–0.472),0.457 (0.429–0.489),0.908 (0.896–0.921) / 0.034 (0.000–0.079) / 0.384 (0.338–0.431),0.869 (0.855–0.882) / 0.066 (0.000–0.145) / 0.437 (0.385–0.489)
Diabetes-H,1000,Random forest,0.370 (0.352–0.388),0.368 (0.355–0.382),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.329 (0.250–0.402),0.953 (0.944–0.962) / 0.000 (0.000–0.000) / 0.152 (0.113–0.190)
Diabetes-H,1000,NAM,0.371 (0.353–0.388),0.368 (0.357–0.380),0.855 (0.842–0.870) / 0.000 (0.000–0.000) / 0.540 (0.429–0.659),0.984 (0.979–0.990) / 0.000 (0.000–0.000) / 0.121 (0.087–0.155)
Diabetes-H,1000,Logistic-KAN,0.384 (0.366–0.402),0.377 (0.365–0.390),0.860 (0.846–0.874) / 0.000 (0.000–0.000) / 0.497 (0.404–0.591),0.977 (0.970–0.983) / 0.000 (0.000–0.000) / 0.155 (0.118–0.195)
Diabetes-H,1000,KAAM,0.384 (0.366–0.402),0.377 (0.365–0.390),0.860 (0.846–0.874) / 0.000 (0.000–0.000) / 0.497 (0.404–0.591),0.977 (0.970–0.983) / 0.000 (0.000–0.000) / 0.155 (0.118–0.195)
Diabetes-H,1000,EBM,0.380 (0.362–0.399),0.375 (0.362–0.388),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.553 (0.442–0.660),0.983 (0.977–0.989) / 0.000 (0.000–0.000) / 0.141 (0.106–0.180)
Diabetes-H,1000,XGBoost,0.401 (0.383–0.418),0.395 (0.379–0.410),0.870 (0.856–0.884) / 0.000 (0.000–0.000) / 0.380 (0.318–0.441),0.937 (0.926–0.948) / 0.000 (0.000–0.000) / 0.247 (0.201–0.293)
Diabetes-H,1000,LightGBM,0.402 (0.385–0.420),0.395 (0.381–0.410),0.869 (0.854–0.883) / 0.000 (0.000–0.000) / 0.412 (0.348–0.482),0.949 (0.939–0.958) / 0.000 (0.000–0.000) / 0.236 (0.192–0.281)
Diabetes-130,1000,MLP,0.328 (0.313–0.344),0.347 (0.334–0.361),0.559 (0.537–0.584) / 0.383 (0.344–0.420) / 0.051 (0.000–0.117),0.738 (0.714–0.761) / 0.292 (0.262–0.321) / 0.011 (0.000–0.024)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2
Obesity class order: 0 / 1 / 2 / 3 / 4 / 5 / 6


##### Multiclass metrics — training size N = 1,688

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [33]:
show_table('multiclass_1688', 'multiclass', size_setting(1688))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Obesity,1688,MLP,0.854 (0.819–0.888),0.856 (0.822–0.889),0.881 (0.784–0.962) / 0.775 (0.649–0.891) / 0.738 (0.625–0.845) / 0.791 (0.680–0.893) / 0.914 (0.844–0.971) / 0.897 (0.821–0.967) / 1.000 (1.000–1.000),0.833 (0.720–0.930) / 0.654 (0.527–0.783) / 0.826 (0.722–0.919) / 0.793 (0.680–0.891) / 0.899 (0.825–0.961) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)
Obesity,1688,Logistic regression,0.709 (0.669–0.751),0.725 (0.688–0.763),0.725 (0.614–0.830) / 0.745 (0.571–0.893) / 0.614 (0.492–0.737) / 0.639 (0.488–0.776) / 0.659 (0.542–0.761) / 0.739 (0.642–0.829) / 0.956 (0.897–1.000),0.887 (0.796–0.964) / 0.361 (0.235–0.488) / 0.656 (0.531–0.774) / 0.518 (0.385–0.652) / 0.686 (0.573–0.794) / 0.983 (0.944–1.000) / 0.986 (0.950–1.000)
Obesity,1688,Random forest,0.976 (0.959–0.989),0.976 (0.959–0.989),0.983 (0.936–1.000) / 0.916 (0.836–0.981) / 0.967 (0.914–1.000) / 1.000 (1.000–1.000) / 0.986 (0.952–1.000) / 0.985 (0.947–1.000) / 1.000 (1.000–1.000),1.000 (1.000–1.000) / 0.950 (0.885–1.000) / 0.982 (0.940–1.000) / 0.932 (0.863–0.985) / 0.986 (0.955–1.000) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)
Obesity,1688,NAM,0.850 (0.815–0.881),0.852 (0.818–0.882),0.960 (0.898–1.000) / 0.749 (0.627–0.855) / 0.827 (0.724–0.917) / 0.728 (0.592–0.850) / 0.782 (0.680–0.878) / 0.937 (0.871–0.986) / 0.984 (0.943–1.000),0.926 (0.854–0.982) / 0.825 (0.722–0.917) / 0.827 (0.726–0.918) / 0.605 (0.490–0.729) / 0.813 (0.714–0.902) / 0.983 (0.945–1.000) / 0.986 (0.950–1.000)
Obesity,1688,Logistic-KAN,0.964 (0.945–0.980),0.964 (0.945–0.980),0.982 (0.940–1.000) / 0.964 (0.906–1.000) / 0.873 (0.782–0.952) / 0.962 (0.909–1.000) / 0.973 (0.929–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000),1.000 (1.000–1.000) / 0.931 (0.860–0.984) / 0.949 (0.889–1.000) / 0.914 (0.837–0.983) / 0.984 (0.952–1.000) / 0.983 (0.945–1.000) / 0.986 (0.950–1.000)
Obesity,1688,KAAM,0.953 (0.933–0.972),0.953 (0.933–0.972),0.962 (0.902–1.000) / 0.979 (0.936–1.000) / 0.833 (0.735–0.917) / 0.943 (0.878–1.000) / 0.986 (0.952–1.000) / 0.985 (0.947–1.000) / 1.000 (1.000–1.000),0.980 (0.939–1.000) / 0.861 (0.760–0.947) / 0.946 (0.883–1.000) / 0.896 (0.808–0.967) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)
Obesity,1688,EBM,0.968 (0.951–0.983),0.968 (0.951–0.983),1.000 (1.000–1.000) / 0.982 (0.938–1.000) / 0.915 (0.833–0.982) / 0.912 (0.836–0.979) / 0.973 (0.929–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000),1.000 (1.000–1.000) / 0.948 (0.881–1.000) / 0.928 (0.857–0.983) / 0.931 (0.857–0.984) / 0.986 (0.952–1.000) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)
Obesity,1688,XGBoost,0.990 (0.980–0.998),0.990 (0.980–0.998),1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.950 (0.891–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.985 (0.947–1.000) / 1.000 (1.000–1.000),1.000 (1.000–1.000) / 0.965 (0.911–1.000) / 1.000 (1.000–1.000) / 0.983 (0.944–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)
Obesity,1688,LightGBM,0.990 (0.980–0.998),0.990 (0.980–0.998),1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.950 (0.892–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.985 (0.947–1.000) / 1.000 (1.000–1.000),1.000 (1.000–1.000) / 0.982 (0.943–1.000) / 1.000 (1.000–1.000) / 0.965 (0.912–1.000) / 1.000 (1.000–1.000) / 1.000 (1.000–1.000) / 0.986 (0.950–1.000)


Obesity class order: 0 / 1 / 2 / 3 / 4 / 5 / 6


##### Multiclass metrics — training size N = 5,000

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [34]:
show_table('multiclass_5000', 'multiclass', size_setting(5000))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,5000,MLP,0.372 (0.354–0.390),0.369 (0.357–0.382),0.855 (0.841–0.869) / 0.000 (0.000–0.000) / 0.419 (0.330–0.508),0.969 (0.962–0.976) / 0.000 (0.000–0.000) / 0.138 (0.101–0.175)
Diabetes-H,5000,Logistic regression,0.440 (0.419–0.464),0.448 (0.426–0.472),0.900 (0.887–0.913) / 0.029 (0.000–0.098) / 0.374 (0.324–0.420),0.876 (0.862–0.889) / 0.022 (0.000–0.068) / 0.445 (0.393–0.497)
Diabetes-H,5000,Random forest,0.423 (0.407–0.440),0.439 (0.420–0.459),0.894 (0.881–0.908) / 0.000 (0.000–0.000) / 0.343 (0.300–0.392),0.857 (0.841–0.871) / 0.000 (0.000–0.000) / 0.459 (0.405–0.516)
Diabetes-H,5000,NAM,0.382 (0.363–0.400),0.375 (0.364–0.388),0.858 (0.844–0.871) / 0.000 (0.000–0.000) / 0.599 (0.493–0.703),0.985 (0.980–0.990) / 0.000 (0.000–0.000) / 0.141 (0.106–0.178)
Diabetes-H,5000,Logistic-KAN,0.381 (0.363–0.401),0.375 (0.364–0.388),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.591 (0.486–0.701),0.985 (0.979–0.990) / 0.000 (0.000–0.000) / 0.141 (0.105–0.179)
Diabetes-H,5000,KAAM,0.381 (0.363–0.401),0.375 (0.364–0.388),0.858 (0.844–0.872) / 0.000 (0.000–0.000) / 0.591 (0.486–0.701),0.985 (0.979–0.990) / 0.000 (0.000–0.000) / 0.141 (0.105–0.179)
Diabetes-H,5000,EBM,0.400 (0.379–0.419),0.389 (0.374–0.403),0.864 (0.850–0.878) / 0.000 (0.000–0.000) / 0.556 (0.467–0.643),0.977 (0.970–0.983) / 0.000 (0.000–0.000) / 0.190 (0.148–0.232)
Diabetes-H,5000,XGBoost,0.405 (0.386–0.423),0.395 (0.380–0.409),0.867 (0.853–0.881) / 0.000 (0.000–0.000) / 0.474 (0.397–0.552),0.963 (0.954–0.970) / 0.000 (0.000–0.000) / 0.222 (0.178–0.264)
Diabetes-H,5000,LightGBM,0.401 (0.382–0.419),0.390 (0.376–0.404),0.864 (0.850–0.877) / 0.000 (0.000–0.000) / 0.525 (0.435–0.612),0.972 (0.964–0.979) / 0.000 (0.000–0.000) / 0.198 (0.158–0.240)
Diabetes-130,5000,MLP,0.363 (0.347–0.379),0.389 (0.377–0.402),0.587 (0.565–0.608) / 0.499 (0.457–0.543) / 0.389 (0.000–0.800),0.863 (0.844–0.880) / 0.295 (0.265–0.325) / 0.011 (0.000–0.024)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2


##### Multiclass metrics — training size N = 10,000

Only multiclass tasks available in this setting are included. Classwise precision and recall follow the printed class order and are separated by `/`.

In [35]:
show_table('multiclass_10000', 'multiclass', size_setting(10000))

Dataset,Training size,Model,Macro-F1,Balanced accuracy,Precision by class,Recall by class
Diabetes-H,10000,MLP,0.387 (0.367–0.405),0.380 (0.366–0.393),0.859 (0.845–0.873) / 0.000 (0.000–0.000) / 0.504 (0.411–0.595),0.974 (0.967–0.981) / 0.000 (0.000–0.000) / 0.165 (0.125–0.203)
Diabetes-H,10000,Logistic regression,0.437 (0.420–0.454),0.449 (0.430–0.468),0.900 (0.887–0.913) / 0.000 (0.000–0.000) / 0.385 (0.338–0.434),0.882 (0.868–0.896) / 0.000 (0.000–0.000) / 0.465 (0.407–0.521)
Diabetes-H,10000,Random forest,0.419 (0.404–0.434),0.450 (0.430–0.467),0.906 (0.892–0.918) / 0.000 (0.000–0.000) / 0.319 (0.282–0.357),0.816 (0.799–0.832) / 0.000 (0.000–0.000) / 0.532 (0.476–0.584)
Diabetes-H,10000,NAM,0.374 (0.356–0.393),0.370 (0.359–0.382),0.856 (0.841–0.869) / 0.000 (0.000–0.000) / 0.614 (0.491–0.726),0.988 (0.982–0.992) / 0.000 (0.000–0.000) / 0.124 (0.089–0.160)
Diabetes-H,10000,Logistic-KAN,0.361 (0.344–0.378),0.362 (0.352–0.373),0.853 (0.839–0.867) / 0.000 (0.000–0.000) / 0.575 (0.444–0.700),0.989 (0.984–0.993) / 0.000 (0.000–0.000) / 0.097 (0.068–0.131)
Diabetes-H,10000,KAAM,0.361 (0.344–0.378),0.362 (0.352–0.373),0.853 (0.839–0.867) / 0.000 (0.000–0.000) / 0.575 (0.444–0.700),0.989 (0.984–0.993) / 0.000 (0.000–0.000) / 0.097 (0.068–0.131)
Diabetes-H,10000,EBM,0.392 (0.373–0.411),0.383 (0.370–0.397),0.861 (0.847–0.875) / 0.000 (0.000–0.000) / 0.558 (0.462–0.651),0.979 (0.973–0.985) / 0.000 (0.000–0.000) / 0.170 (0.130–0.212)
Diabetes-H,10000,XGBoost,0.394 (0.376–0.414),0.385 (0.372–0.399),0.863 (0.849–0.876) / 0.000 (0.000–0.000) / 0.518 (0.433–0.614),0.974 (0.967–0.981) / 0.000 (0.000–0.000) / 0.181 (0.142–0.223)
Diabetes-H,10000,LightGBM,0.383 (0.366–0.401),0.376 (0.365–0.389),0.858 (0.845–0.872) / 0.000 (0.000–0.000) / 0.562 (0.462–0.670),0.982 (0.976–0.988) / 0.000 (0.000–0.000) / 0.147 (0.111–0.184)
Diabetes-130,10000,MLP,0.359 (0.341–0.375),0.388 (0.376–0.401),0.585 (0.565–0.607) / 0.513 (0.463–0.562) / 0.298 (0.091–0.538),0.899 (0.883–0.914) / 0.247 (0.219–0.277) / 0.018 (0.004–0.035)


Diabetes-H class order: 0 / 1 / 2
Diabetes-130 class order: 0 / 1 / 2


## LaTeX export

After displaying a table, use its name to print LaTeX with the same values and bold styling. The example exports the main reference table. To export another table, replace `main_reference` with its table key, such as `main_100`, `calibration_reference` or `multiclass_1000`. The notebook does not write an export file automatically.

In [36]:
# print(latex_table('main_reference'))